# LeetCode 1861 — Rotating the Box bằng Python

Link đề: https://leetcode.com/problems/rotating-the-box/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers, Matrix

Skeleton LeetCode:

```python
class Solution:
    def rotateTheBox(self, boxGrid: List[List[str]]) -> List[List[str]]:
        ...
```

## 1. Mục tiêu học

1. Tách bài thành hai việc độc lập: **trọng lực** (đá rơi) và **xoay ma trận 90° theo chiều kim đồng hồ** — và nhận ra có thể làm trọng lực **trước** khi xoay, trên từng hàng.
2. Luyện kỹ thuật **hai con trỏ "vị trí ghi"** để dồn phần tử về một phía trong `O(n)` mỗi hàng, có **vật cản** chia hàng thành các đoạn độc lập.
3. Nắm công thức xoay: `new[j][m−1−i] = old[i][j]`, và cách viết gọn `zip(*grid[::-1])`.
4. Phân biệt xoay **thuận** và **ngược** chiều kim đồng hồ, kích thước kết quả `n × m`.
5. Viết mô phỏng trọng lực "từng bước" làm brute force để kiểm chứng.

## 2. Đề bài & ví dụ

Cho ma trận ký tự `boxGrid` kích thước `m × n` biểu diễn **mặt bên** của một cái hộp. Mỗi ô là:
- `'#'` — một viên đá,
- `'*'` — vật cản cố định,
- `'.'` — ô trống.

Hộp được **xoay 90° theo chiều kim đồng hồ**, khiến một số viên đá rơi do trọng lực. Mỗi viên đá rơi xuống cho đến khi chạm vật cản, viên đá khác hoặc đáy hộp. Trọng lực không ảnh hưởng vật cản, và quán tính khi xoay không làm đá dịch ngang. Đề đảm bảo trước khi xoay, mỗi viên đá đã nằm yên (trên vật cản, đá khác hoặc đáy).

Trả về ma trận `n × m` biểu diễn hộp sau khi xoay.

**Ví dụ 1:** `[["#",".","#"]]` → `[["."],["#"],["#"]]`.

**Ví dụ 2:** `[["#",".","*","."],["#","#","*","."]]` → `[["#","."],["#","#"],["*","*"],[".","."]]`.

**Ví dụ 3:** `[["#","#","*",".","*","."],["#","#","#","*",".","."],["#","#","#",".","#","."]]` → `[[".","#","#"],[".","#","#"],["#","#","*"],["#","*","."],["#",".","*"],["#",".","."]]`.

**Ràng buộc:** `1 ≤ m, n ≤ 500`; mỗi ô là `'#'`, `'*'` hoặc `'.'`.

In [1]:
import random
import time
from typing import List

def G(rows):                       # tiện viết lưới bằng chuỗi
    return [list(r) for r in rows]

examples = [
    (G(["#.#"]), G([".", "#", "#"])),
    (G(["#.*.", "##*."]), G(["#.", "##", "**", ".."])),
    (G(["##*.*.", "###*..", "###.#."]), G([".##", ".##", "##*", "#*.", "#.*", "#.."])),
]
def show(grid):
    return "\n".join(" ".join(r) for r in grid)
for g, exp in examples:
    print(show(g), "\n->\n" + show(exp), "\n")

# . # 
->
.
#
# 

# . * .
# # * . 
->
# .
# #
* *
. . 

# # * . * .
# # # * . .
# # # . # . 
->
. # #
. # #
# # *
# * .
# . *
# . . 



## 3. Kiến thức nền

### 3.1 Xoay ma trận 90° theo chiều kim đồng hồ

Ô `(i, j)` của ma trận `m × n` đi đến ô `(j, m−1−i)` của ma trận `n × m`. Cách nhớ: **hàng trên cùng** trở thành **cột phải cùng**; cột trái cùng (đọc từ dưới lên) trở thành hàng trên cùng.

In [2]:
A = [[1, 2, 3],
     [4, 5, 6]]                  # m = 2, n = 3
m, n = len(A), len(A[0])
R = [[None] * m for _ in range(n)]
for i in range(m):
    for j in range(n):
        R[j][m - 1 - i] = A[i][j]
print(R, "| zip(*A[::-1]) =", [list(r) for r in zip(*A[::-1])])

[[4, 1], [5, 2], [6, 3]] | zip(*A[::-1]) = [[4, 1], [5, 2], [6, 3]]


### 3.2 Trọng lực sau khi xoay = "dồn sang phải" trước khi xoay

Sau khi xoay thuận chiều kim đồng hồ, hướng "xuống" của hộp mới tương ứng với hướng **sang phải** của hộp cũ (cột cuối cùng của hàng cũ thành đáy). Vậy: **trong mỗi hàng ban đầu, dồn đá sang phải**, dừng ở vật cản; rồi xoay.

In [3]:
row = list("#.#*.#.")
print("hàng gốc       :", "".join(row))
print("đá dồn phải    :", "".join("*".join("".join(sorted(seg, reverse=True)) for seg in "".join(row).split("*"))))

hàng gốc       : #.#*.#.
đá dồn phải    : .##*..#


### 3.3 Mẹo Python: thứ tự ký tự

`ord('#') = 35 < ord('*') = 42 < ord('.') = 46`. Vì thế `sorted(seg, reverse=True)` trên một đoạn không có `'*'` sẽ đặt mọi `'.'` trước, mọi `'#'` sau — chính là "dồn đá sang phải".

In [4]:
print([(c, ord(c)) for c in "#*."], "| sorted('#.#..', reverse=True) =", "".join(sorted("#.#..", reverse=True)))

[('#', 35), ('*', 42), ('.', 46)] | sorted('#.#..', reverse=True) = ...##


### 3.4 Tạo ma trận 2 chiều đúng cách

`[['.'] * m] * n` tạo `n` **tham chiếu đến cùng một list** — sửa một hàng là sửa tất cả. Phải dùng `[['.'] * m for _ in range(n)]`.

In [5]:
bad = [["."] * 2] * 3; bad[0][0] = "#"
good = [["."] * 2 for _ in range(3)]; good[0][0] = "#"
print("bad :", bad); print("good:", good)

bad : [['#', '.'], ['#', '.'], ['#', '.']]
good: [['#', '.'], ['.', '.'], ['.', '.']]


## 4. Ý tưởng nền tảng / trực giác

Xét từng hàng của hộp ban đầu. Các vật cản `'*'` chia hàng thành **các đoạn**. Trong mỗi đoạn, khi hộp xoay, đá "rơi" về phía **cuối đoạn** (phía phải), chồng lên nhau. Số đá trong đoạn giữ nguyên, chỉ vị trí thay đổi: `k` viên đá chiếm `k` ô cuối của đoạn.

Tính tay **Ví dụ 3**, từng hàng:

| hàng | các đoạn (tách bởi `*`) | sau khi dồn phải |
|---|---|---|
| `##*.*.` | `##` / `.` / `.` | `##*.*.` |
| `###*..` | `###` / `..` | `###*..` |
| `###.#.` | `###.#.` (4 đá, 6 ô) | `..####` |

Sau đó xoay: hàng 0 thành cột phải cùng, hàng 2 thành cột trái cùng.

In [6]:
grid, exp = examples[2]
pushed = []
for r in grid:
    s = "*".join("".join(sorted(seg, reverse=True)) for seg in "".join(r).split("*"))
    pushed.append(list(s)); print("".join(r), "->", s)
rot = [list(x) for x in zip(*pushed[::-1])]
print(show(rot)); assert rot == exp

##*.*. -> ##*.*.
###*.. -> ###*..
###.#. -> ..####
. # #
. # #
# # *
# * .
# . *
# . .


## 5. Cách 1 — Brute Force / baseline: xoay trước, rồi mô phỏng trọng lực từng bước

Xoay ma trận, sau đó lặp: quét tìm mọi viên đá có ô ngay bên dưới trống, cho nó rơi **một ô**; lặp đến khi không còn viên nào rơi được. Đúng "nghĩa đen" của đề nhất.

In [7]:
def rotate_cw(grid):
    m, n = len(grid), len(grid[0])
    res = [[None] * m for _ in range(n)]
    for i in range(m):
        for j in range(n):
            res[j][m - 1 - i] = grid[i][j]
    return res

def rotate_box_bruteforce(boxGrid):
    g = rotate_cw(boxGrid)
    rows, cols = len(g), len(g[0])
    moved = True
    while moved:
        moved = False
        for r in range(rows - 2, -1, -1):              # từ dưới lên để rơi nhanh hơn
            for c in range(cols):
                if g[r][c] == "#" and g[r + 1][c] == ".":
                    g[r][c], g[r + 1][c] = ".", "#"
                    moved = True
    return g

for g, exp in examples:
    got = rotate_box_bruteforce(g); assert got == exp
print("Brute force khớp 3 ví dụ")

Brute force khớp 3 ví dụ


**Độ phức tạp:** mỗi lượt quét `O(mn)`. Quét từ dưới lên, một viên đá rơi **tối đa một ô mỗi lượt** (ô nó rơi vào đã được quét trong lượt này), mà có thể phải rơi tới `n` ô ⇒ tệ nhất `n` lượt ⇒ `O(mn · n)` ≈ `1.25·10⁸` với `500 × 500` — quá chậm trong Python.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (đổi thứ tự):** trọng lực sau xoay = dồn đá sang phải **trong hàng gốc**. Các hàng độc lập với nhau (đá không đổi hàng gốc — "không dịch ngang" trong hộp mới).

**Insight 2 (vị trí ghi):** duyệt hàng **từ phải sang trái**, giữ `empty` = vị trí **thấp nhất còn trống** (phải nhất) mà đá có thể rơi tới.
- Gặp `'*'`: đá không thể vượt qua ⇒ `empty = j − 1`.
- Gặp `'#'`: chuyển đá đến `empty` (đặt `row[j] = '.'`, `row[empty] = '#'`), rồi `empty −= 1`.
- Gặp `'.'`: không làm gì.

**Invariant:** tại mọi thời điểm, các ô từ `empty + 1` đến vật cản (hoặc mép phải) gần nhất đều đã là **đá** đã ổn định; ô `empty` chưa có đá.

**Insight 3 (xoay khi ghi):** có thể ghi trực tiếp vào ma trận kết quả: ô `(i, j)` của hàng đã dồn đi vào `res[j][m−1−i]`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Dồn phải bằng con trỏ ghi, rồi xoay

In [8]:
def push_right(row):
    row = row[:]                                   # không sửa input
    empty = len(row) - 1
    for j in range(len(row) - 1, -1, -1):
        if row[j] == "*":
            empty = j - 1
        elif row[j] == "#":
            row[j] = "."
            row[empty] = "#"
            empty -= 1
    return row

def rotate_box_two_pointers(boxGrid):
    return rotate_cw([push_right(r) for r in boxGrid])

for g, exp in examples:
    assert rotate_box_two_pointers(g) == exp
print(push_right(list("#.#*.#.")), "| khớp 3 ví dụ")

['.', '#', '#', '*', '.', '.', '#'] | khớp 3 ví dụ


Thứ tự hai lệnh `row[j] = "."` rồi `row[empty] = "#"` quan trọng: khi viên đá **đã** ở đúng chỗ (`empty == j`), đặt `'.'` trước rồi `'#'` sau sẽ giữ nó lại; làm ngược thứ tự sẽ xóa mất viên đá.

**Độ phức tạp Cách 2:** `O(mn)` thời gian, `O(mn)` bộ nhớ cho kết quả.

### 7.2 Cách 3 — Đếm đá theo đoạn (split / sort)

In [9]:
def rotate_box_segments(boxGrid):
    pushed = []
    for r in boxGrid:
        segs = "".join(r).split("*")
        new = "*".join("." * seg.count(".") + "#" * seg.count("#") for seg in segs)
        pushed.append(new)
    return [list(col) for col in zip(*pushed[::-1])]

for g, exp in examples:
    assert rotate_box_segments(g) == exp
print("Cách 3 khớp 3 ví dụ")

Cách 3 khớp 3 ví dụ


Mỗi đoạn giữa hai vật cản chỉ cần biết **số đá** và **số ô trống**: ô trống trước, đá sau. Dùng `count` thay vì `sorted` để `O(độ dài)` thay vì `O(độ dài · log)`.

**Độ phức tạp Cách 3:** `O(mn)`.

### 7.3 Cách 4 — Ghi thẳng vào ma trận đã xoay (một lượt)

Khởi tạo `res` toàn `'.'` kích thước `n × m`. Với hàng `i`, duyệt từ phải sang trái như Cách 2 nhưng ghi `'*'` và `'#'` trực tiếp vào `res[·][m−1−i]`.

In [10]:
class Solution:
    def rotateTheBox(self, boxGrid: List[List[str]]) -> List[List[str]]:
        m, n = len(boxGrid), len(boxGrid[0])
        res = [["."] * m for _ in range(n)]
        for i in range(m):
            col = m - 1 - i                           # hàng i -> cột m-1-i
            empty = n - 1                             # vị trí đá có thể rơi tới (theo chỉ số j)
            for j in range(n - 1, -1, -1):
                c = boxGrid[i][j]
                if c == "*":
                    res[j][col] = "*"
                    empty = j - 1
                elif c == "#":
                    res[empty][col] = "#"
                    empty -= 1
        return res

for g, exp in examples:
    got = Solution().rotateTheBox(g); assert got == exp
print("Cách 4 khớp 3 ví dụ")

Cách 4 khớp 3 ví dụ


Ưu điểm: không sửa input, không cần ma trận trung gian, mỗi ô xử lý đúng một lần. Ô `'.'` không phải ghi vì `res` đã khởi tạo bằng `'.'`.

**Độ phức tạp Cách 4:** `O(mn)` thời gian, `O(mn)` cho kết quả (bắt buộc).

## 8. TRACE từng bước

Trace con trỏ `empty` của Cách 2 trên hàng `"#.#*.#."` (ví dụ tự dựng, chỉ để minh họa):

In [11]:
def trace_push(row):
    row = list(row); empty = len(row) - 1
    for j in range(len(row) - 1, -1, -1):
        c = row[j]
        if c == "*":
            empty = j - 1; act = f"vật cản -> empty = {empty}"
        elif c == "#":
            row[j] = "."; row[empty] = "#"; act = f"đá chuyển {j} -> {empty}"; empty -= 1
        else:
            act = "trống, bỏ qua"
        print(f"j={j} '{c}': {act:<24} hàng = {''.join(row)}")
    return "".join(row)

assert trace_push("#.#*.#.") == ".##*..#" == "".join(push_right(list("#.#*.#.")))

j=6 '.': trống, bỏ qua            hàng = #.#*.#.
j=5 '#': đá chuyển 5 -> 6         hàng = #.#*..#
j=4 '.': trống, bỏ qua            hàng = #.#*..#
j=3 '*': vật cản -> empty = 2     hàng = #.#*..#
j=2 '#': đá chuyển 2 -> 2         hàng = #.#*..#
j=1 '.': trống, bỏ qua            hàng = #.#*..#
j=0 '#': đá chuyển 0 -> 1         hàng = .##*..#


| j | ký tự | hành động | empty sau | hàng |
|---|---|---|---|---|
| 6 | `.` | bỏ qua | 6 | `#.#*.#.` |
| 5 | `#` | 5 → 6 | 5 | `#.#*..#` |
| 4 | `.` | bỏ qua | 5 | `#.#*..#` |
| 3 | `*` | reset | 2 | `#.#*..#` |
| 2 | `#` | 2 → 2 (giữ nguyên) | 1 | `#.#*..#` |
| 1 | `.` | bỏ qua | 1 | `#.#*..#` |
| 0 | `#` | 0 → 1 | 0 | `.##*..#` |

Kết quả `.##*..#`: đoạn trái `#.#` có 2 đá ⇒ `.##`; đoạn phải `.#.` có 1 đá ⇒ `..#`. Chú ý ở `j = 2` viên đá đã ở đúng chỗ (`empty == j`) nên chỉ cần giữ nguyên.

Trace tiếp phép xoay trên **Ví dụ 2**: hàng 0 `"#.*."` dồn thành `".#*."`, hàng 1 `"##*."` giữ nguyên; xoay ⇒ hàng 1 thành cột 0, hàng 0 thành cột 1.

In [12]:
g, exp = examples[1]
p = [push_right(r) for r in g]
print("sau dồn:", ["".join(r) for r in p])
for j in range(len(g[0])):
    print(f"res[{j}] = [hàng1[{j}], hàng0[{j}]] =", [p[1][j], p[0][j]])
assert rotate_cw(p) == exp

sau dồn: ['.#*.', '##*.']
res[0] = [hàng1[0], hàng0[0]] = ['#', '.']
res[1] = [hàng1[1], hàng0[1]] = ['#', '#']
res[2] = [hàng1[2], hàng0[2]] = ['*', '*']
res[3] = [hàng1[3], hàng0[3]] = ['.', '.']


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 4** — ghi thẳng vào ma trận đã xoay bằng con trỏ `empty`.

In [13]:
class Solution:
    def rotateTheBox(self, boxGrid: List[List[str]]) -> List[List[str]]:
        m, n = len(boxGrid), len(boxGrid[0])
        res = [["."] * m for _ in range(n)]
        for i in range(m):
            col = m - 1 - i
            empty = n - 1
            for j in range(n - 1, -1, -1):
                c = boxGrid[i][j]
                if c == "*":
                    res[j][col] = "*"
                    empty = j - 1
                elif c == "#":
                    res[empty][col] = "#"
                    empty -= 1
        return res

for g, exp in examples:
    got = Solution().rotateTheBox(g); print(got == exp); assert got == exp

True
True
True


## 10. Vì sao thuật toán đúng

**Tách trọng lực và xoay.** Sau khi xoay, cột `m−1−i` của hộp mới chính là hàng `i` của hộp cũ, đọc theo thứ tự `j = 0..n−1` từ trên xuống. "Xuống" trong hộp mới ⇔ `j` tăng ⇔ "sang phải" trong hàng cũ. Đá không dịch ngang ⇒ mỗi viên ở yên trong cột của nó ⇒ bài toán độc lập cho từng hàng cũ: dồn đá sang phải.

**Trạng thái cuối của một hàng là duy nhất.** Trong một đoạn giữa hai vật cản (hoặc mép), đá rơi cho đến khi bị chặn ⇒ cuối cùng `k` viên đá chiếm đúng `k` ô **cuối** của đoạn (nếu còn ô trống ngay dưới một viên đá, nó chưa dừng). Đá không vượt qua vật cản.

**Con trỏ `empty`.** Quy nạp khi duyệt từ phải sang trái trong một đoạn: sau khi xử lý các ô `j..r` (với `r` là cuối đoạn), các viên đá đã gặp chiếm đúng các ô `empty+1..r`. Gặp đá mới ⇒ đặt nó vào `empty` (ô trống phải nhất còn lại) ⇒ bất biến giữ. Gặp vật cản ⇒ bắt đầu đoạn mới với `empty = j − 1`. Khi kết thúc đoạn, đá chiếm đúng `k` ô cuối ⇒ khớp trạng thái duy nhất. ∎

### 10.1 Kiểm chứng thực nghiệm: "hai thứ tự" cho cùng kết quả

Mô phỏng trọng lực **sau** khi xoay (brute force) và "dồn phải **trước** rồi xoay" cho cùng kết quả trên lưới ngẫu nhiên — xác nhận Insight 1. Đồng thời kiểm tra số đá và vị trí vật cản được bảo toàn.

In [14]:
def rand_grid(rng, m, n, p=(0.35, 0.15)):
    return [[("#" if (x := rng.random()) < p[0] else "*" if x < p[0] + p[1] else ".") for _ in range(n)] for _ in range(m)]

rng = random.Random(1010)
for _ in range(300):
    g = rand_grid(rng, rng.randint(1, 6), rng.randint(1, 7))
    a = rotate_box_bruteforce(g); b = rotate_box_two_pointers(g)
    assert a == b
    assert sum(r.count("#") for r in a) == sum(r.count("#") for r in g)
    rot_raw = rotate_cw(g)
    assert all((rot_raw[i][j] == "*") == (a[i][j] == "*") for i in range(len(a)) for j in range(len(a[0])))
print("Trọng lực-sau-xoay = dồn-phải-rồi-xoay; số đá và vật cản bảo toàn (300 lưới)")

Trọng lực-sau-xoay = dồn-phải-rồi-xoay; số đá và vật cản bảo toàn (300 lưới)


## 11. Phân tích Time / Space Complexity

Gọi `m, n ≤ 500` (tối đa `2.5·10⁵` ô).

| Cách | Time | Space |
|---|---|---|
| 1. Xoay + mô phỏng từng bước | `O(mn · n)` tệ nhất | `O(mn)` |
| 2. Dồn phải (con trỏ) + xoay | `O(mn)` | `O(mn)` |
| 3. Đếm theo đoạn + `zip` | `O(mn)` | `O(mn)` |
| 4. Ghi thẳng vào kết quả | `O(mn)` | `O(mn)` (kết quả) |

In [15]:
rng = random.Random(1861)
big = rand_grid(rng, 500, 500)
worst = [["#"] * 250 + ["."] * 250 for _ in range(80)]      # nhiều đá phải rơi xa
for name, f in [("two_pointers", rotate_box_two_pointers), ("segments", rotate_box_segments), ("submit", Solution().rotateTheBox)]:
    t = time.perf_counter(); f(big); print(f"{name:<13} 500x500: {(time.perf_counter() - t) * 1000:.0f} ms")
t = time.perf_counter(); rotate_box_bruteforce(worst); print(f"brute 80x500 (đá phải rơi 250 ô): {(time.perf_counter() - t) * 1000:.0f} ms")

two_pointers  500x500: 24 ms
segments      500x500: 10 ms
submit        500x500: 14 ms


brute 80x500 (đá phải rơi 250 ô): 438 ms


## 12. Bộ test

In [16]:
ALL = {"brute": rotate_box_bruteforce, "two_pointers": rotate_box_two_pointers,
       "segments": rotate_box_segments, "submit": Solution().rotateTheBox}
def run_all(g, brute=True):
    snapshot = [r[:] for r in g]
    res = {k: f(g) for k, f in ALL.items() if brute or k != "brute"}
    assert g == snapshot, "input bị sửa"
    vals = list(res.values())
    assert all(v == vals[0] for v in vals), (g, res)
    return vals[0]

for g, exp in examples:
    assert run_all(g) == exp
# ví dụ tự dựng (đối chiếu lại đề gốc) — kỳ vọng kiểm bằng brute force trong run_all
edge = [(G(["#"]), G(["#"])), (G(["."]), G(["."])), (G(["*"]), G(["*"])),
        (G(["#", "."]), G(["." + "#"])), (G(["*#."]), G(["*", ".", "#"])),
        (G(["##."]), G([".", "#", "#"])), (G([".#", "#."]), G(["..", "##"]))]
for g, exp in edge:
    assert run_all(g) == exp, (g, run_all(g))
rng = random.Random(18610)
for _ in range(500):
    run_all(rand_grid(rng, rng.randint(1, 8), rng.randint(1, 8), (rng.random() * 0.6, rng.random() * 0.3)))
for _ in range(3):
    run_all(rand_grid(rng, 500, 500), brute=False)
print("Ví dụ, 7 edge case, 500 lưới nhỏ (đối chiếu brute), 3 lưới 500x500: OK")

Ví dụ, 7 edge case, 500 lưới nhỏ (đối chiếu brute), 3 lưới 500x500: OK


### 12.1 Kiểm tra tính chất

- Kích thước kết quả là `n × m`.
- **Lũy đẳng:** áp dụng "trọng lực" lần nữa lên kết quả (không xoay) không đổi gì.
- Mỗi cột kết quả, giữa hai vật cản: mọi `'#'` nằm **dưới** mọi `'.'`.

In [17]:
def settle_down(g):                      # trọng lực trên lưới đã xoay (không xoay thêm)
    g = [r[:] for r in g]; moved = True
    while moved:
        moved = False
        for r in range(len(g) - 1):
            for c in range(len(g[0])):
                if g[r][c] == "#" and g[r + 1][c] == ".":
                    g[r][c], g[r + 1][c] = ".", "#"; moved = True
    return g

rng = random.Random(121)
f = Solution().rotateTheBox
for _ in range(300):
    m, n = rng.randint(1, 7), rng.randint(1, 7)
    g = rand_grid(rng, m, n); r = f(g)
    assert len(r) == n and all(len(row) == m for row in r)
    assert settle_down(r) == r
    for c in range(m):
        col = "".join(r[i][c] for i in range(n))
        assert all("#." not in seg for seg in col.split("*"))
print("3 tính chất đúng trên 300 lưới")

3 tính chất đúng trên 300 lưới


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Xoay **ngược** chiều kim đồng hồ

In [18]:
def wrong_ccw(boxGrid):
    pushed = [push_right(r) for r in boxGrid]
    return [list(c) for c in zip(*pushed)][::-1]           # SAI: đây là xoay ngược chiều

g, exp = examples[1]
print("sai :\n" + show(wrong_ccw(g)), "\nđúng:\n" + show(exp))

sai :
. .
* *
# #
. # 
đúng:
# .
# #
* *
. .


**Vì sao sai:** `zip(*A)` là **chuyển vị**; chuyển vị rồi đảo thứ tự hàng = xoay **ngược** chiều. Xoay **thuận** là đảo thứ tự hàng **trước** rồi chuyển vị: `zip(*A[::-1])`. **Sửa:** dùng công thức `res[j][m−1−i] = A[i][j]` và kiểm tra trên ví dụ nhỏ `[[1,2,3],[4,5,6]]`.

### Lỗi 2 — Dồn đá sang **trái** (nhầm hướng trọng lực)

In [19]:
def wrong_push_left(boxGrid):
    pushed = []
    for r in boxGrid:
        s = "*".join("#" * seg.count("#") + "." * seg.count(".") for seg in "".join(r).split("*"))
        pushed.append(list(s))
    return rotate_cw(pushed)

g, exp = examples[0]
print("sai :", wrong_push_left(g), "| đúng:", exp)

sai : [['#'], ['#'], ['.']] | đúng: [['.'], ['#'], ['#']]


**Vì sao sai:** sau khi xoay thuận chiều, **phải** của hàng cũ thành **đáy**. **Sửa:** ô trống trước, đá sau trong mỗi đoạn.

### Lỗi 3 — Quên vật cản: dồn toàn bộ đá của hàng về mép phải

In [20]:
def wrong_ignore_obstacles(boxGrid):
    pushed = []
    for r in boxGrid:
        k = r.count("#")
        row = ["*" if c == "*" else "." for c in r]
        j = len(row) - 1
        while k:                                         # SAI: bỏ qua '*' và đặt đá "xuyên" vật cản
            if row[j] == ".":
                row[j] = "#"; k -= 1
            j -= 1
        pushed.append(row)
    return rotate_cw(pushed)

g, exp = examples[1]
print("sai :\n" + show(wrong_ignore_obstacles(g)), "\nđúng:\n" + show(exp))

sai :
. .
# .
* *
# # 
đúng:
# .
# #
* *
. .


**Vì sao sai:** đá bên trái vật cản bị "dịch chuyển xuyên tường" sang bên phải. Vật cản chia hàng thành các đoạn **độc lập**. **Sửa:** reset `empty = j − 1` khi gặp `'*'`.

### Lỗi 4 — Một lượt quét trái → phải, đổi chỗ `'#'` với `'.'` kế bên

In [21]:
def wrong_single_pass(boxGrid):
    pushed = []
    for r in boxGrid:
        row = r[:]
        for j in range(len(row) - 1):
            if row[j] == "#" and row[j + 1] == ".":
                row[j], row[j + 1] = ".", "#"                # SAI: một lượt không đủ
        pushed.append(row)
    return rotate_cw(pushed)

g = G(["#.#."])          # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", ["".join(r) for r in wrong_single_pass(g)], "| đúng:", ["".join(r) for r in Solution().rotateTheBox(g)],
      "| brute:", ["".join(r) for r in rotate_box_bruteforce(g)])

sai : ['.', '#', '.', '#'] | đúng: ['.', '.', '#', '#'] | brute: ['.', '.', '#', '#']


**Vì sao sai:** viên đá đầu tiên đuổi theo viên thứ hai, nhưng khi viên thứ hai di chuyển thì viên đầu đã được quét qua. Lặp lại tới khi ổn định thì đúng nhưng `O(n²)` mỗi hàng. **Sửa:** quét **từ phải sang trái** với con trỏ `empty`.

### Lỗi 5 — Ghi đè đá trong Cách 2 do sai thứ tự lệnh

In [22]:
def wrong_order_push(row):
    row = row[:]; empty = len(row) - 1
    for j in range(len(row) - 1, -1, -1):
        if row[j] == "*":
            empty = j - 1
        elif row[j] == "#":
            row[empty] = "#"
            row[j] = "."                                 # SAI: khi empty == j sẽ xóa mất đá
            empty -= 1
    return row

print("sai :", "".join(wrong_order_push(list("..##"))), "| đúng:", "".join(push_right(list("..##"))))

sai : .... | đúng: ..##


**Vì sao sai:** khi viên đá đã ở đúng chỗ (`empty == j`), đặt `'#'` rồi đặt `'.'` vào cùng ô sẽ xóa nó. **Sửa:** đặt `'.'` trước, `'#'` sau (hoặc ghi vào ma trận kết quả mới như Cách 4).

### Lỗi 6 — Tạo kết quả bằng `[['.'] * m] * n`

In [23]:
def wrong_alias(boxGrid):
    m, n = len(boxGrid), len(boxGrid[0])
    res = [["."] * m] * n                                  # SAI: n tham chiếu cùng một list
    for i in range(m):
        col, empty = m - 1 - i, n - 1
        for j in range(n - 1, -1, -1):
            c = boxGrid[i][j]
            if c == "*": res[j][col] = "*"; empty = j - 1
            elif c == "#": res[empty][col] = "#"; empty -= 1
    return res

g, exp = examples[1]
print("sai :", wrong_alias(g), "\nđúng:", exp)

sai : [['#', '#'], ['#', '#'], ['#', '#'], ['#', '#']] 
đúng: [['#', '.'], ['#', '#'], ['*', '*'], ['.', '.']]


**Vì sao sai:** mọi hàng của `res` là cùng một đối tượng ⇒ ghi vào một hàng là ghi vào tất cả. **Sửa:** list comprehension `[["."] * m for _ in range(n)]`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Xoay rồi trọng lực" ⇒ **trọng lực trước, xoay sau** — đưa về bài 1D trên từng hàng.
- "Dồn phần tử về một phía, có vật cản" ⇒ con trỏ ghi, duyệt **ngược** hướng dồn, reset ở vật cản (giống 283 Move Zeroes có vật cản).
- Xoay thuận: `zip(*A[::-1])`; xoay ngược: `zip(*A)[::-1]`; luôn thử với ma trận `2 × 3`.
- Kích thước kết quả đổi `m × n` → `n × m`.
- Tránh `[[x] * m] * n`.

## 15. Biến thể của bài

1. **48 Rotate Image**: xoay ma trận vuông **tại chỗ** (chuyển vị + đảo hàng).
2. **Xoay ngược chiều kim đồng hồ** rồi trọng lực ⇒ đá dồn về **trái** của hàng gốc (Bài tập 4).
3. **Trọng lực theo 4 hướng** (game 2048 không gộp ô, Candy Crush 723) — viết `gravity(grid, dir)` tổng quát (Bài tập 5).
4. **Xoay `k` lần**, mỗi lần áp trọng lực — sau lần đầu, hình dạng tiếp theo được quyết định bởi các lần dồn liên tiếp.
5. **2132/1914 Cyclically Rotating a Grid** — xoay từng lớp viền.

### 15.1 Code biến thể: xoay tại chỗ ma trận vuông (48 Rotate Image) để so với công thức

In [24]:
def rotate_in_place(M):
    n = len(M)
    for i in range(n):
        for j in range(i + 1, n):
            M[i][j], M[j][i] = M[j][i], M[i][j]          # chuyển vị
    for row in M:
        row.reverse()                                     # đảo mỗi hàng
    return M

rng = random.Random(48)
for _ in range(100):
    k = rng.randint(1, 6)
    A = [[rng.randint(0, 9) for _ in range(k)] for _ in range(k)]
    assert rotate_in_place([r[:] for r in A]) == rotate_cw(A) == [list(r) for r in zip(*A[::-1])]
print("Xoay tại chỗ = công thức res[j][m-1-i] = zip(*A[::-1]) trên 100 ma trận vuông")

Xoay tại chỗ = công thức res[j][m-1-i] = zip(*A[::-1]) trên 100 ma trận vuông


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Xoay + mô phỏng | Nghĩa đen của đề | `O(mn·n)` | `O(mn)` | Kiểm chứng |
| 2. Con trỏ `empty` + xoay | Dồn phải từng hàng | `O(mn)` | `O(mn)` | Dễ hiểu, tách bạch |
| 3. Đếm theo đoạn | `split('*')` + đếm | `O(mn)` | `O(mn)` | Viết nhanh bằng Python |
| 4. Ghi thẳng kết quả | Con trỏ + công thức xoay | `O(mn)` | `O(mn)` | **Bản nộp** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `["#"]`; (b) `["*#."]`; (c) `["##."]`; (d) `[".#", "#."]`.

**Bài 2 (tính tay):** Dồn phải hàng `"#.*#.#..*#"` rồi cho biết hàng đó trở thành **cột nào** của kết quả nếu nó là hàng `i = 0` trong lưới `m = 3` hàng.

**Bài 3 (điền TODO):** Hoàn thiện `push_right_todo(row)` bằng con trỏ `empty`.

**Bài 4 (tự viết lại):** Viết `rotate_box_ccw(grid)`: xoay **ngược** chiều kim đồng hồ rồi áp trọng lực; đối chiếu brute force (xoay ngược + mô phỏng).

**Bài 5 (mini challenge):** Viết `gravity(grid, d)` với `d ∈ {"down","up","left","right"}`: đá dồn theo hướng `d`, dừng ở vật cản/đá/mép; `O(mn)`. Đối chiếu mô phỏng từng bước. Kiểm tra: `rotateTheBox(g) == gravity(rotate_cw(g), "down")`.

In [25]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def push_right_todo(row):
    row = row[:]; empty = len(row) - 1
    for j in range(len(row) - 1, -1, -1):
        # TODO: '*' -> empty = j - 1; '#' -> row[j] = '.', row[empty] = '#', empty -= 1
        pass
    return row

In [26]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def rotate_box_ccw(grid):
    # TODO: xoay ngược: res[n-1-j][i] = grid[i][j]; "xuống" lúc này ứng với "trái" của hàng gốc
    pass

def gravity(grid, d):
    # TODO: với "down"/"up" xử lý theo cột, "left"/"right" theo hàng; dùng con trỏ empty duyệt ngược hướng dồn
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: dồn phải từng hàng. Mức 2: xoay, kích thước `n × m`. Mức 3: (a) `[["#"]]`; (b) `[["*"],["."],["#"]]`; (c) `[["."],["#"],["#"]]`; (d) `[[".","."],["#","#"]]`.

**Bài 2.** Mức 1: tách đoạn: `"#."`, `"#.#.."`, `"#"`. Mức 2: dồn: `".#"`, `"...##"`, `"#"` ⇒ `".#*...##*#"`. Mức 3: hàng `i = 0` với `m = 3` thành cột `m − 1 − i = 2` (cột phải cùng).

**Bài 3.** Mức 1: duyệt từ phải. Mức 2: reset ở vật cản. Mức 3: đặt `'.'` trước `'#'`.

**Bài 4.** Mức 1: xoay ngược đưa **trái** của hàng gốc xuống đáy. Mức 2: dồn trái từng hàng (ô đá trước, ô trống sau trong mỗi đoạn). Mức 3: `res[n−1−j][i] = row[j]`.

**Bài 5.** Mức 1: "right" = Cách 2; "left" = đối xứng. Mức 2: "down"/"up" = áp lên cột (chuyển vị, dồn, chuyển vị lại). Mức 3: mô phỏng brute: lặp đổi chỗ đá với ô trống kề theo hướng `d` đến khi ổn định.

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1, 2, 3
f = Solution().rotateTheBox
b1 = [f(G(x)) for x in (["#"], ["*#."], ["##."], [".#", "#."])]
print("Bài 1:", b1)
assert b1 == [[["#"]], [["*"], ["."], ["#"]], [["."], ["#"], ["#"]], [[".", "."], ["#", "#"]]]
b2 = "".join(push_right(list("#.*#.#..*#")))
print("Bài 2:", b2); assert b2 == ".#*...##*#"
def push_right_todo(row):
    row = row[:]; empty = len(row) - 1
    for j in range(len(row) - 1, -1, -1):
        if row[j] == "*":
            empty = j - 1
        elif row[j] == "#":
            row[j] = "."; row[empty] = "#"; empty -= 1
    return row
rng = random.Random(3)
for _ in range(200):
    r = rand_grid(rng, 1, rng.randint(1, 10))[0]
    assert push_right_todo(r) == push_right(r)
print("Bài 3: OK")

Bài 1: [[['#']], [['*'], ['.'], ['#']], [['.'], ['#'], ['#']], [['.', '.'], ['#', '#']]]
Bài 2: .#*...##*#
Bài 3: OK


In [28]:
# Bài 4, 5
def rotate_ccw(g):
    m, n = len(g), len(g[0])
    res = [[None] * m for _ in range(n)]
    for i in range(m):
        for j in range(n):
            res[n - 1 - j][i] = g[i][j]
    return res

def push_line(line, toward_end=True):
    seq = line if toward_end else line[::-1]
    out = "*".join("." * s.count(".") + "#" * s.count("#") for s in "".join(seq).split("*"))
    out = list(out)
    return out if toward_end else out[::-1]

def transpose(g):
    return [list(r) for r in zip(*g)]

def gravity(grid, d):
    if d in ("right", "left"):
        return [push_line(r, d == "right") for r in grid]
    return transpose([push_line(r, d == "down") for r in transpose(grid)])

def gravity_brute(grid, d):
    g = [r[:] for r in grid]; dr, dc = {"down": (1, 0), "up": (-1, 0), "left": (0, -1), "right": (0, 1)}[d]
    moved = True
    while moved:
        moved = False
        for i in range(len(g)):
            for j in range(len(g[0])):
                a, b = i + dr, j + dc
                if 0 <= a < len(g) and 0 <= b < len(g[0]) and g[i][j] == "#" and g[a][b] == ".":
                    g[i][j], g[a][b] = ".", "#"; moved = True
    return g

def rotate_box_ccw(grid):
    return gravity(rotate_ccw(grid), "down")

rng = random.Random(5)
for _ in range(300):
    g = rand_grid(rng, rng.randint(1, 6), rng.randint(1, 6))
    for d in ("down", "up", "left", "right"):
        assert gravity(g, d) == gravity_brute(g, d)
    assert Solution().rotateTheBox(g) == gravity(rotate_cw(g), "down")
    assert rotate_box_ccw(g) == gravity_brute(rotate_ccw(g), "down") == rotate_ccw([push_line(r, False) for r in g])
print("Bài 4, 5: OK |", rotate_box_ccw(G(["#.#"])))

Bài 4, 5: OK | [['.'], ['#'], ['#']]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (ghi thẳng vào kết quả):**

```python
res = [["."] * m for _ in range(n)]
for i in range(m):
    col, empty = m - 1 - i, n - 1
    for j in range(n - 1, -1, -1):          # ngược hướng dồn
        if box[i][j] == "*": res[j][col] = "*"; empty = j - 1
        elif box[i][j] == "#": res[empty][col] = "#"; empty -= 1
return res
```

**Checklist:**
- [ ] Xoay **thuận** chiều: `res[j][m−1−i] = old[i][j]` (kích thước `n × m`)?
- [ ] Trọng lực = dồn **phải** trong hàng gốc?
- [ ] Reset con trỏ ở vật cản `'*'`?
- [ ] Không xóa đá khi `empty == j` (thứ tự lệnh)?
- [ ] Không dùng `[[...] * m] * n`?